# Hugging Face (Transformers / Datasets / Hub)

A refresher on the **Hugging Face stack** — the de-facto registry and toolkit for pretrained models, datasets, and the glue code (`transformers`, `datasets`, `huggingface_hub`) that turns "I want a working NLP/vision/audio model" into three lines of Python.

**Domain:** AI/ML Tooling  ·  **from study list**  ·  **runnable:** yes

## 1. What & Why

**Hugging Face is "npm/PyPI for models" plus the libraries to use them.** Three pieces matter:

- **The Hub** — a Git-LFS-backed registry of ~1M+ pretrained **models**, **datasets**, and **Spaces** (hosted demo apps), each a versioned repo you pull by string id like `"distilbert-base-uncased"`.
- **`transformers`** — a unified Python API over thousands of model architectures (BERT, GPT, Llama, ViT, Whisper, …). The same `AutoModel` / `AutoTokenizer` / `pipeline` interface works regardless of architecture or backend (PyTorch, and increasingly others).
- **`datasets`** — a memory-mapped (Apache Arrow) dataset library that loads, streams, and transforms data too big for RAM, with a `.map()`/`.filter()` API and one-line access to Hub datasets.

**The problem it solves.** Before Hugging Face, using a state-of-the-art model meant hunting down a research repo, matching exact dependency versions, reverse-engineering the author's preprocessing, and porting checkpoints by hand. The Hub standardizes *distribution* (one `from_pretrained(id)` call) and `transformers` standardizes the *API* (one interface across architectures), so swapping `bert` for `roberta` is a string change, not a rewrite.

**Reach for it when** you want to *use* or *fine-tune* a pretrained model (text classification, embeddings, translation, ASR, image classification, LLM inference), need a quick baseline, or want to share a model/dataset. **Don't** reach for it when you're training a novel architecture from scratch (you're back to raw [PyTorch](pytorch.ipynb)), doing classical/tabular ML ([scikit-learn](scikit-learn.ipynb)), or serving a single fixed model at extreme scale where a dedicated server (vLLM, TGI, ONNX Runtime) earns its keep.

## 2. Mental Model

**Think of the Hub as a package registry, and `transformers` as a universal adapter.**

```
            Hugging Face Hub  (versioned Git+LFS repos, addressed by string id)
            ┌───────────────┬───────────────┬───────────────┐
            │    Models     │   Datasets    │    Spaces     │
            └───────┬───────┴───────┬───────┴───────────────┘
                    │ from_pretrained(id)  │ load_dataset(id)
                    ▼                       ▼
        ┌─────────────────────────┐   ┌──────────────────┐
        │      transformers       │   │     datasets     │
        │                         │   │  (Arrow-backed)  │
        │  raw text ─▶ Tokenizer  │   │  .map / .filter  │
        │            ─▶  Model    │   │  .train_test_... │
        │            ─▶ logits    │   └──────────────────┘
        │   pipeline() wraps all three steps into one call │
        └─────────────────────────┘
```

Two layers of abstraction, pick by how much control you need:

1. **`pipeline("task")`** — highest level. Hides tokenize → model → decode. Great for inference and demos.
2. **`AutoTokenizer` + `AutoModel`** — mid level. You own the tokenize/forward/post-process steps. Needed for fine-tuning, custom heads, batching control.

The `Auto*` classes are the magic: they read the repo's `config.json`, see `"model_type": "bert"`, and instantiate the right concrete class for you — so your code never hard-codes the architecture.

## 3. Key Concepts

- **Model id / repo** — a string like `"distilbert-base-uncased"` or `"org/name"` addressing a versioned Hub repo. Pin a specific commit with `revision=`.
- **`from_pretrained(id)` / `save_pretrained(dir)`** — the universal load/save contract for models, tokenizers, and configs. Downloads are cached under `~/.cache/huggingface` (override with `HF_HOME`).
- **Tokenizer** — converts text ↔ integer ids the model expects, including subword splitting (WordPiece/BPE), special tokens (`[CLS]`, `[SEP]`), padding, truncation, and attention masks. **A model and its tokenizer are a matched pair** — always load both from the *same* id.
- **`Auto*` classes** — `AutoTokenizer`, `AutoModel`, `AutoModelForSequenceClassification`, … dispatch to the correct architecture from the repo config. Use the task-specific `AutoModelFor…` so you get the right head.
- **`pipeline(task)`** — end-to-end inference helper for a named task (`"sentiment-analysis"`, `"summarization"`, `"automatic-speech-recognition"`, …).
- **`datasets.Dataset`** — an Arrow table; columnar, memory-mapped, lazily transformed via `.map()`. `DatasetDict` holds named splits (train/validation/test). `streaming=True` iterates without downloading the whole thing.
- **`Trainer` / `TrainingArguments`** — a batteries-included training loop (logging, checkpointing, eval, mixed precision, multi-GPU). The high-level alternative to hand-writing a PyTorch loop.
- **Safetensors** — the safe, fast checkpoint format (no arbitrary-code pickle) that the Hub now defaults to.
- **Hub auth** — a token (`HF_TOKEN`) needed to download gated/private repos or to push. Public models need none.

## 4. Setup

All three libraries are plain `pip` installs. `transformers` needs a backend — install PyTorch alongside it (this notebook uses CPU-only torch). `datasets` is lightweight (Arrow + a few helpers).

```bash
pip install transformers datasets
# A backend is required for running models — CPU-only torch is enough here:
pip install torch --index-url https://download.pytorch.org/whl/cpu

# Optional: authenticate to pull gated/private repos or to push your own.
#   huggingface-cli login          # or set the HF_TOKEN env var
```

In a notebook you'd run `%pip install transformers datasets`. The cell below just verifies the install and reports versions — no network, no downloads.

In [1]:
import os
import transformers
import datasets
import torch

print(f"transformers : {transformers.__version__}")
print(f"datasets     : {datasets.__version__}")
print(f"torch        : {torch.__version__}")

# Hub downloads are cached here (override with HF_HOME). Anything that hits the
# network is gated later behind this flag so the notebook runs fully offline.
RUN_DOWNLOADS = bool(os.getenv("HF_RUN_DOWNLOADS"))
print(f"cache dir    : {os.path.expanduser(os.getenv('HF_HOME', '~/.cache/huggingface'))}")
print(f"HF_RUN_DOWNLOADS set: {RUN_DOWNLOADS}")

/Users/danieldekerlegand/Development/ai-tutor/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


transformers : 5.12.1
datasets     : 5.0.0
torch        : 2.12.1
cache dir    : /Users/danieldekerlegand/.cache/huggingface
HF_RUN_DOWNLOADS set: False


## 5. Worked Examples

Four examples, ordered from fully-offline to network-gated:

1. **`datasets`** — build, transform (`.map`), and split a dataset (offline, real output).
2. **Tokenization** — train a tiny WordPiece tokenizer to *see* subword splitting (offline).
3. **`AutoTokenizer` + `AutoModel`** — the real Hub load/encode/forward path (gated behind `HF_RUN_DOWNLOADS`).
4. **`pipeline`** — three-line task inference (gated behind `HF_RUN_DOWNLOADS`).

### Example 1 — `datasets`: load, map, split

A `Dataset` is an Arrow table. Here we build one from memory (you'd normally `load_dataset("imdb")` from the Hub), add a derived column with `.map()`, then carve out a train/test split — the everyday data-wrangling loop.

In [2]:
from datasets import Dataset

reviews = {
    "text": [
        "I loved this film, an absolute masterpiece",
        "boring and far too long, I want my money back",
        "gorgeous visuals and a tight script",
        "the worst two hours of my life",
    ],
    "label": [1, 0, 1, 0],   # 1 = positive, 0 = negative
}
ds = Dataset.from_dict(reviews)
print(ds)
print("features:", ds.features)
print("row 0   :", ds[0])

# .map applies a function over every row (batched/parallel for big data).
ds = ds.map(lambda ex: {"n_words": len(ex["text"].split())})
print("\nword counts:", ds["n_words"])

# Reproducible split into named train/test sets.
split = ds.train_test_split(test_size=0.5, seed=0)
print("\nsplit:", {k: v.num_rows for k, v in split.items()})

Dataset({
    features: ['text', 'label'],
    num_rows: 4
})
features: {'text': Value('string'), 'label': Value('int64')}
row 0   : {'text': 'I loved this film, an absolute masterpiece', 'label': 1}


Map:   0%|          | 0/4 [00:00<?, ? examples/s]

Map: 100%|██████████| 4/4 [00:00<00:00, 1523.82 examples/s]


word counts: Column([7, 10, 6, 7])

split: {'train': 2, 'test': 2}


### Example 2 — Tokenization: see the subwords

Models don't see text; they see integer ids of **subword** tokens. To make that concrete without any download, we train a tiny WordPiece tokenizer (the same algorithm BERT uses) on our four sentences. Watch how rare words get split into `##`-prefixed continuation pieces — that's how a fixed vocabulary covers an open-ended language.

In [3]:
from tokenizers import Tokenizer
from tokenizers.models import WordPiece
from tokenizers.trainers import WordPieceTrainer
from tokenizers.pre_tokenizers import Whitespace

tok = Tokenizer(WordPiece(unk_token="[UNK]"))
tok.pre_tokenizer = Whitespace()
trainer = WordPieceTrainer(vocab_size=60,
                           special_tokens=["[UNK]", "[CLS]", "[SEP]", "[PAD]"])
tok.train_from_iterator(reviews["text"], trainer)

enc = tok.encode("I loved this masterpiece")
print("vocab size :", tok.get_vocab_size())
print("tokens     :", enc.tokens)   # note the ## subword continuation pieces
print("ids        :", enc.ids)
print("decoded    :", tok.decode(enc.ids))




vocab size : 60
tokens     : ['I', 'lo', '##v', '##e', '##d', 'th', '##is', 'm', '##a', '##st', '##e', '##r', '##p', '##i', '##e', '##c', '##e']
ids        : [5, 52, 46, 38, 34, 54, 56, 17, 33, 58, 38, 29, 41, 30, 38, 42, 38]
decoded    : I lo ##v ##e ##d th ##is m ##a ##st ##e ##r ##p ##i ##e ##c ##e


### Example 3 — `AutoTokenizer` + `AutoModel` (Hub, gated)

The real path: load a *matched* tokenizer+model from a Hub id, tokenize to tensors, and run a forward pass to get hidden states. This downloads weights, so it's gated behind `HF_RUN_DOWNLOADS`; the code shape is exactly what you'd run.

In [4]:
if RUN_DOWNLOADS:
    from transformers import AutoTokenizer, AutoModel

    model_id = "prajjwal1/bert-tiny"            # ~17 MB toy BERT — CPU-friendly
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    model = AutoModel.from_pretrained(model_id)

    batch = tokenizer(
        ["Hugging Face makes transformers easy", "tiny but mighty"],
        padding=True, truncation=True, return_tensors="pt",
    )
    print("input_ids shape:", tuple(batch["input_ids"].shape))

    with torch.no_grad():
        out = model(**batch)
    # last_hidden_state: (batch, seq_len, hidden_dim) contextual embeddings.
    print("last_hidden_state:", tuple(out.last_hidden_state.shape))
else:
    print("Skipped (set HF_RUN_DOWNLOADS=1 to run). Call shape it would execute:")
    print('  tok = AutoTokenizer.from_pretrained("prajjwal1/bert-tiny")')
    print('  model = AutoModel.from_pretrained("prajjwal1/bert-tiny")')
    print('  out = model(**tok(["hello"], return_tensors="pt"))')

Skipped (set HF_RUN_DOWNLOADS=1 to run). Call shape it would execute:
  tok = AutoTokenizer.from_pretrained("prajjwal1/bert-tiny")
  model = AutoModel.from_pretrained("prajjwal1/bert-tiny")
  out = model(**tok(["hello"], return_tensors="pt"))


### Example 4 — `pipeline`: task inference in three lines (gated)

`pipeline` wraps tokenize → model → decode for a named task. This is the fastest way to a working model and the right tool for inference and demos. Gated because it downloads a fine-tuned checkpoint on first use.

In [5]:
if RUN_DOWNLOADS:
    from transformers import pipeline

    clf = pipeline("sentiment-analysis")        # downloads a default SST-2 model
    print(clf(["I loved this film", "I want my money back"]))
else:
    print("Skipped (set HF_RUN_DOWNLOADS=1 to run). It would execute:")
    print('  clf = pipeline("sentiment-analysis")')
    print('  clf(["I loved this film", "I want my money back"])')
    print("  -> [{'label': 'POSITIVE', 'score': 0.99...},")
    print("      {'label': 'NEGATIVE', 'score': 0.99...}]")

Skipped (set HF_RUN_DOWNLOADS=1 to run). It would execute:
  clf = pipeline("sentiment-analysis")
  clf(["I loved this film", "I want my money back"])
  -> [{'label': 'POSITIVE', 'score': 0.99...},
      {'label': 'NEGATIVE', 'score': 0.99...}]


## 6. Gotchas & Pitfalls

- **Mismatched tokenizer and model.** Loading a tokenizer from one id and a model from another silently produces garbage — different vocabularies. Always `from_pretrained` *both* from the same id.
- **Wrong `AutoModel` head.** Bare `AutoModel` returns hidden states, not predictions. For classification use `AutoModelForSequenceClassification`; for generation `AutoModelForCausalLM`. The wrong head means a missing/random classification layer.
- **Forgetting `model.eval()` + `torch.no_grad()` at inference.** Same trap as raw [PyTorch](pytorch.ipynb): dropout stays on and you waste memory building a graph you won't backprop.
- **Unpinned revisions.** A model id resolves to *latest* by default; the repo owner can update weights and silently change your results. Pin `revision="<commit-sha>"` for reproducibility.
- **First call is slow / needs network.** `from_pretrained` downloads on a cache miss. In CI or offline, pre-cache and set `HF_HUB_OFFLINE=1`, or you'll get surprise network hangs.
- **Gated & private repos 401/403.** Some models (e.g. Llama) require accepting a license and an `HF_TOKEN`. A 401 usually means "not authenticated," not "doesn't exist."
- **Padding/truncation forgotten when batching.** Variable-length sequences must be padded to a common length (`padding=True`) and capped (`truncation=True`) or tensor creation fails / OOMs on a stray long input.
- **`datasets.map` without `batched=True`** is slow on large data — batch it, and use `num_proc` for multiprocessing. Also remember `.map` returns a *new* dataset; it isn't in place.
- **Pickle checkpoints.** Old `.bin` checkpoints are pickles (arbitrary-code-execution risk from untrusted repos). Prefer **safetensors** (now the default) and be wary of untrusted `trust_remote_code=True`.
- **Disk fills up.** The cache (`~/.cache/huggingface`) grows fast with large models. Clean it or relocate via `HF_HOME`.

## 7. When to Use vs Alternatives

| Tool | Use it when | Trade-off vs Hugging Face |
|------|-------------|---------------------------|
| **HF Transformers + Hub** | Use/fine-tune pretrained models across text/vision/audio; quick baselines; share models | Heavyweight; abstractions can hide detail; not optimal for max-throughput serving |
| **Raw [PyTorch](pytorch.ipynb)** | Novel architectures from scratch, full control of the loop | You write tokenization, loading, training loop yourself |
| **[scikit-learn](scikit-learn.ipynb)** | Classical/tabular ML (trees, linear, SVM) | Not deep learning; no pretrained transformers |
| **vLLM / TGI** | High-throughput **LLM serving** in production (paged attention, batching) | Inference only; you still get the weights *from* the Hub |
| **[ONNX Runtime](onnx-runtime.ipynb)** | Optimized cross-platform inference, edge/mobile deploy | Export step; less flexible than eager `transformers` |
| **OpenAI / Anthropic APIs** | Want a top frontier LLM with zero infra, pay per token | Closed weights, no fine-tune control, data leaves your box |
| **[PyTorch Lightning](pytorch-lightning.ipynb)** | Custom training loops with less boilerplate, non-transformer models | Not model-zoo-centric; you bring the model |

**Rule of thumb:** want a pretrained model fast → `pipeline`. Fine-tuning → `AutoModelFor…` + `Trainer` (or Lightning). Production LLM serving at scale → pull weights from the Hub, serve with vLLM/TGI. Tabular/classical → scikit-learn.

## 8. Resources

- **Transformers docs** — <https://huggingface.co/docs/transformers/index> (the API reference; `Auto*`, `pipeline`, `Trainer`).
- **Datasets docs** — <https://huggingface.co/docs/datasets/index> (loading, `.map`, streaming, Arrow internals).
- **The Hub** — <https://huggingface.co/docs/hub/index> (repos, revisions, gated models, auth, Spaces).
- **NLP Course (free)** — <https://huggingface.co/learn/nlp-course> (the canonical hands-on on-ramp to the whole stack).
- **`huggingface_hub` client** — <https://huggingface.co/docs/huggingface_hub/index> (programmatic download/upload, `hf_hub_download`).
- **Related notebooks in this library:** [PyTorch](pytorch.ipynb), [PyTorch Lightning](pytorch-lightning.ipynb), [scikit-learn](scikit-learn.ipynb), [ONNX Runtime](onnx-runtime.ipynb).